# 29 · Designing good tools
Level L3 · Part 4 · core · ~35 min · builds on 27, 28

## Goal
Rewrite a vague `do_stuff` tool into tools a model can use well: clear names and descriptions, narrow
inputs, small high-signal results with pagination and truncation, a calculator that never runs code,
and a `write_file` that cannot leave Scout's folder. Then count what the tool definitions cost in tokens.

## The idea
The model only knows a tool through its name, description and schema, and only learns what happened
through the result. So a tool is a small user interface, and its user is the model. A good tool does
one job, says when to use it, takes few and narrow inputs, and returns only what helps the next step.
It also makes misuse hard or impossible (**poka-yoke**, Japanese for "mistake-proofing"): a file tool
that *cannot* write outside its folder, a calculator that *cannot* run code. And every tool definition
is sent on every call, so it costs tokens. Think of a well-labelled light switch versus a panel of
unlabelled buttons: same wiring, very different results.

## Picture
```mermaid
flowchart LR
    V["do_stuff(action, data)<br/>vague, returns everything"] -->|rewrite| G
    subgraph G[good tools]
      S["search_notes<br/>paginated, truncated"]
      C["calculator<br/>AST, no eval"]
      W["write_file<br/>confined to one folder"]
    end
    G --> T["tool definitions<br/>cost tokens on every call"]
```

## Step by step
### 1. Start the recorder and set up
The recorder first. Without a key, the Gemini calls replay from
`../data/llm_cassettes/29_writing-good-tools.yaml`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("29_writing-good-tools")

LLM mode: replay (29_writing-good-tools.yaml)


Setup from lessons 26 to 28: Scout's notes, the scripted fake model and its history check.
Scout's notes have grown: a 30-day habitat log was imported, plus one long briefing. That is what
makes result size matter.

In [2]:
import ast, inspect, json, operator, os
from collections.abc import Callable
from pathlib import Path
from typing import Annotated, get_args, get_origin, get_type_hints

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
BRIEFING = {"id": 7, "text": "Mars mission briefing: " + "check suits, check air, check water. " * 40}
LOG = [{"id": 100 + d, "text": f"Mars habitat log, day {d}: routine checks, all systems normal."}
       for d in range(1, 31)]
ALL_NOTES = NOTES + [BRIEFING] + LOG
print(len(ALL_NOTES), "notes,", sum(len(n["text"]) for n in ALL_NOTES), "characters")

37 notes, 3610 characters


`ScriptedModel` and its history check, unchanged from lesson 26.

In [3]:
def check_history(messages: list[dict]) -> None:
    issued = set()
    for m in messages:
        issued |= {c["id"] for c in m.get("tool_calls", [])}
        if m["role"] == "tool" and m["call_id"] not in issued:
            raise ValueError(f"tool result for unknown call id {m['call_id']!r}")

class ScriptedModel:
    def __init__(self, script: list):
        self.script, self.calls, self.next_id = script, 0, 1

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        check_history(messages)
        self.calls += 1
        if self.calls > 50:
            raise RuntimeError("model called 50 times: the loop never stopped")
        reply = self.script[min(self.calls, len(self.script)) - 1]
        reply = reply(messages) if callable(reply) else json.loads(json.dumps(reply))
        for call in reply.get("tool_calls", []):
            call["id"], self.next_id = f"call_{self.next_id}", self.next_id + 1
        return {"role": "assistant", **reply}

Lesson 27's schema builder and registry, unchanged.

In [4]:
JSON_TYPES = {str: "string", int: "integer", float: "number", bool: "boolean"}

def json_type(hint) -> dict:
    if hint in JSON_TYPES:
        return {"type": JSON_TYPES[hint]}
    if get_origin(hint) is list:
        (item,) = get_args(hint)
        return {"type": "array", "items": json_type(item)}
    raise TypeError(f"no JSON Schema type for {hint!r}")

def schema_from_function(fn) -> dict:
    hints = get_type_hints(fn, include_extras=True)
    properties, required = {}, []
    for name, param in inspect.signature(fn).parameters.items():
        if name not in hints:
            raise TypeError(f"{fn.__name__}: parameter {name!r} has no type hint")
        hint, note = hints[name], None
        if get_origin(hint) is Annotated:
            hint, note = get_args(hint)[0], get_args(hint)[1]
        properties[name] = json_type(hint) | ({"description": note} if note else {})
        if param.default is inspect.Parameter.empty:
            required.append(name)
    if not inspect.getdoc(fn):
        raise TypeError(f"{fn.__name__}: needs a docstring; it becomes the tool description")
    return {"name": fn.__name__, "description": inspect.getdoc(fn),
            "parameters": {"type": "object", "properties": properties, "required": required}}

The registry class from lesson 27.

In [5]:
class ToolRegistry:
    def __init__(self):
        self.functions: dict[str, Callable] = {}
        self.schemas: list[dict] = []

    def tool(self, fn):
        schema = schema_from_function(fn)
        if schema["name"] in self.functions:
            raise ValueError(f"tool {schema['name']!r} is already registered")
        self.functions[schema["name"]] = fn
        self.schemas.append(schema)
        return fn

    def call(self, name: str, args: dict):
        return self.functions[name](**args)

Lesson 28's `run_tool` (never raises) and its `run_agent`, unchanged.

In [6]:
def run_tool(call: dict, tools: dict) -> dict:
    fn = tools.get(call["name"])
    if fn is None:
        return {"content": f"unknown tool '{call['name']}'. Available tools: {', '.join(tools)}.",
                "is_error": True}
    try:
        return {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:
        return {"content": f"{type(e).__name__}: {e}", "is_error": True}

And the loop that sends error results back.

In [7]:
def run_agent(question: str, model, tools: dict, schemas: list[dict], max_steps: int = 6) -> dict:
    messages, errors = [{"role": "user", "text": question}], 0
    for step in range(1, max_steps + 1):
        reply = model(messages, schemas)
        messages.append(reply)
        calls = reply.get("tool_calls", [])
        if not calls:
            print(f"step {step}: answer")
            return {"answer": reply["text"], "stop": "end_of_turn", "steps": step,
                    "errors": errors, "messages": messages}
        for call in calls:
            result = run_tool(call, tools)
            errors += result["is_error"]
            print(f"step {step}: {call['name']}({call['args']}) -> "
                  f"{'ERROR ' if result['is_error'] else ''}{result['content'][:70]}")
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"], **result})
    return {"answer": None, "stop": "step_limit", "steps": max_steps, "errors": errors, "messages": messages}

### 2. A vague tool: `do_stuff`
Here is a tool someone wrote in a hurry. One function does three jobs, chosen by a string. Read it as
the model would: only the name, the description and two strings called `action` and `data`.

In [8]:
def do_stuff(action: str, data: str) -> str:
    """Does stuff with notes."""
    if action == "find":
        return str([n for n in ALL_NOTES if data in n["text"]])
    if action == "calc":
        return str(eval(data))                                   # runs any Python the model sends
    if action == "save":
        name, text = data.split("|", 1)
        Path(name).write_text(text)                              # writes anywhere on disk
        return "ok"
    return "unknown action"

print(json.dumps(schema_from_function(do_stuff), indent=1))

{
 "name": "do_stuff",
 "description": "Does stuff with notes.",
 "parameters": {
  "type": "object",
  "properties": {
   "action": {
    "type": "string"
   },
   "data": {
    "type": "string"
   }
  },
  "required": [
   "action",
   "data"
  ]
 }
}


Nothing in that schema says which actions exist, what `data` should look like, or what comes back.
Now call it the way a model might. Lower-case `mars` finds nothing (the search is case-sensitive), and
`Mars` returns every match as one long Python string.

In [9]:
for query in ["mars", "Mars"]:
    out = do_stuff("find", query)
    print(f"find {query!r}: {len(out):5} characters | {out[:90]}")

find 'mars':     2 characters | []
find 'Mars':  4315 characters | [{'id': 1, 'text': 'Mars has two small moons, Phobos and Deimos.'}, {'id': 3, 'text': 'A d


Over 4,000 characters, most of it the same log line and a repeated briefing, to answer
"how long is a day on Mars?". All of it would go into the history and be paid for on every later call.

### 3. Rewrite: a narrow, high-signal `search_notes`
One job, a name that says it, a description that says what comes back and when to use it. Results
are **high-signal**: only `id` and `text`, long texts shortened, at most a page of them, plus the
total and the next page number. That is **pagination**: the model asks for more only if it needs it.

In [10]:
def shorten(text: str, limit: int = 120) -> str:
    """Cut long text and say how much was cut, so the model knows it is not the whole note."""
    return text if len(text) <= limit else text[:limit] + f"... [+{len(text) - limit} chars]"

print(shorten(BRIEFING["text"]))

Mars mission briefing: check suits, check air, check water. check suits, check air, check water. check suits, check air,... [+1383 chars]


Now the tool itself. Bad input gets an actionable error (lesson 28). An out-of-range `max_results`
is not an error: it is clamped to 1..10, so the model cannot ask for all 37 notes at once.

In [11]:
registry = ToolRegistry()

@registry.tool
def search_notes(query: Annotated[str, "a word or short phrase, e.g. 'Mars'; case is ignored"],
                 max_results: Annotated[int, "notes per page, 1 to 10"] = 5,
                 page: Annotated[int, "page number, starting at 1"] = 1) -> dict:
    """Search the user's notes for a word or phrase (case-insensitive). Returns one page of matching
    notes as {id, text} (long texts shortened), the total number of matches and next_page (null when
    there are no more). Use it before answering any question about the notes."""
    q = query.strip().lower()
    if not q or page < 1:
        raise ValueError("query must not be empty and page starts at 1, e.g. query='Mars', page=1")
    hits = [n for n in ALL_NOTES if q in n["text"].lower()]
    size = min(max(max_results, 1), 10)                         # poka-yoke: clamp, don't fail
    start = (page - 1) * size
    return {"total": len(hits), "page": page,
            "results": [{"id": n["id"], "text": shorten(n["text"])} for n in hits[start:start + size]],
            "next_page": page + 1 if start + size < len(hits) else None}

Same question as before. Compare the size, and notice the day-length note is on page 1.

In [12]:
first = search_notes("mars")
print("total:", first["total"], "| next_page:", first["next_page"], "|",
      len(json.dumps(first)), "characters")
for r in first["results"]:
    print(" ", r["id"], r["text"][:80])
print("last page:", search_notes("Mars", max_results=10, page=4)["next_page"])

total: 34 | next_page: 2 | 516 characters
  1 Mars has two small moons, Phobos and Deimos.
  3 A day on Mars lasts about 24 hours and 39 minutes.
  5 The rover Perseverance landed on Mars in February 2021.
  7 Mars mission briefing: check suits, check air, check water. check suits, check a
  101 Mars habitat log, day 1: routine checks, all systems normal.
last page: None


### 4. A size limit on every result
A tool you did not write well (or a huge file) can still return a giant result. One cap in
`run_tool` protects every tool: past `MAX_RESULT_CHARS` the result is **truncated**, with a note
saying so and how to get less. This is the one change to lesson 28's `run_tool`.

In [13]:
MAX_RESULT_CHARS = 1500
run_tool_28 = run_tool

def run_tool(call: dict, tools: dict) -> dict:
    result = run_tool_28(call, tools)
    extra = len(result["content"]) - MAX_RESULT_CHARS
    if extra > 0:
        result["content"] = (result["content"][:MAX_RESULT_CHARS] + f"... [truncated {extra} "
                             "characters: ask for a smaller page or a narrower query]")
    return result

big = run_tool({"name": "do_stuff", "args": {"action": "find", "data": "Mars"}}, {"do_stuff": do_stuff})
print(len(big["content"]), "characters |", big["content"][-85:])

1575 characters | uits, chec... [truncated 2817 characters: ask for a smaller page or a narrower query]


`run_agent` looks `run_tool` up by name each time, so the loop now uses the capped version with no
other change.

### 5. Why `eval` must never be a tool
`do_stuff("calc", ...)` passes model text to `eval`. Model text can come from a web page or a note
(lesson 51), so this is running a stranger's code. Here a harmless "calculation" imports a module
and calls the operating system.

In [14]:
sneaky = "__import__('os').getpid() > 0"
print("do_stuff returned", do_stuff("calc", sneaky), "-> it imported a module and called the OS")

do_stuff returned True -> it imported a module and called the OS


The fix is **safe evaluation**: parse the text into a syntax tree with `ast` and walk it, allowing only
numbers and a fixed list of operators. Lesson 26's calculator did + - * /. This one adds `%`, `**` and
a leading minus, and caps the exponent so `9**9**9` cannot freeze your machine.

In [15]:
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.Mod: operator.mod, ast.Pow: operator.pow}
HOW = "use numbers, + - * / % ** and brackets, e.g. '24*60+39'"

def evaluate(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):  # not True/False
        return node.value
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return -evaluate(node.operand)
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        left, right = evaluate(node.left), evaluate(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 100:
            raise ValueError(f"exponent {right} is too large (max 100)")
        return OPS[type(node.op)](left, right)
    raise ValueError(f"'{ast.unparse(node)[:40]}' is not plain arithmetic: {HOW}")

The tool wraps it: a length limit, a readable error for a syntax error, and a description that tells
the model to use it rather than doing sums in its head.

In [16]:
@registry.tool
def calculator(expression: Annotated[str, "arithmetic only, e.g. '(24*60+39)/60'"]) -> float:
    """Evaluate an arithmetic expression: numbers, + - * / % ** and brackets. Returns the number.
    Use it for every calculation instead of doing arithmetic yourself."""
    if len(expression) > 200:
        raise ValueError("expression too long (max 200 characters); split it into steps")
    try:
        tree = ast.parse(expression, mode="eval")
    except SyntaxError:
        raise ValueError(f"'{expression}' is not valid arithmetic: {HOW}") from None
    return evaluate(tree.body)

for expr in ["24*60+39", "-2**10 % 7", sneaky, "9**9**9", "2^^3"]:
    print(f"{expr:38}", run_tool({"name": "calculator", "args": {"expression": expr}}, registry.functions))

24*60+39                               {'content': '1479', 'is_error': False}
-2**10 % 7                             {'content': '5', 'is_error': False}
__import__('os').getpid() > 0          {'content': "ValueError: '__import__('os').getpid() > 0' is not plain arithmetic: use numbers, + - * / % ** and brackets, e.g. '24*60+39'", 'is_error': True}
9**9**9                                {'content': 'ValueError: exponent 387420489 is too large (max 100)', 'is_error': True}
2^^3                                   {'content': "ValueError: '2^^3' is not valid arithmetic: use numbers, + - * / % ** and brackets, e.g. '24*60+39'", 'is_error': True}


The sneaky call and the exponent bomb come back as error results; nothing ran.

### 6. `write_file` that cannot leave its folder
`do_stuff("save", ...)` writes wherever the name points. A name like `../x.txt` climbs out of the
folder: that is **path traversal**. Watch the naive join do it (we delete the stray file right after).

In [17]:
SCOUT_DIR = Path("../data/scout_files")
SCOUT_DIR.mkdir(parents=True, exist_ok=True)
for old in SCOUT_DIR.glob("*.txt"):                             # start each run with an empty folder
    old.unlink()

escaped = SCOUT_DIR / "../escaped.txt"                          # naive: just join the name
escaped.write_text("should never be here")
print("naive join wrote:", escaped.resolve().relative_to(Path("..").resolve()))
escaped.unlink()

naive join wrote: data/escaped.txt


The safe version **resolves** the path first (turning `..`, absolute paths and symlinks into the real
location) and checks that the real parent is Scout's folder. It also refuses to replace a file unless
the model sets `overwrite`: deleting work by accident is made impossible by default.

In [18]:
@registry.tool
def write_file(filename: Annotated[str, "a plain file name, e.g. 'mars_day.txt'"],
               text: Annotated[str, "the full text to save"],
               overwrite: Annotated[bool, "true to replace an existing file"] = False) -> str:
    """Save text to a file in Scout's folder. Returns 'saved N characters to NAME'.
    Refuses an existing file unless overwrite is true."""
    target = (SCOUT_DIR / filename).resolve()
    if target.parent != SCOUT_DIR.resolve():
        raise ValueError(f"'{filename}' is outside Scout's folder. Use a plain file name, "
                         f"e.g. '{Path(filename).name or 'notes.txt'}'.")
    if target.exists() and not overwrite:
        raise FileExistsError(f"'{filename}' already exists. Pick a new name or set overwrite to true.")
    target.write_text(text)
    return f"saved {len(text)} characters to {target.name}"

Try it with good names, traversal, an absolute path and a second write to the same name.

In [19]:
for name in ["mars_day.txt", "../escaped.txt", "/tmp/escaped.txt", "a/../../escaped.txt", "mars_day.txt"]:
    result = run_tool({"name": "write_file", "args": {"filename": name, "text": "1479 minutes"}},
                      registry.functions)
    print(f"{name:22}", result["is_error"], "|", result["content"][:75])
print("folder:", sorted(p.name for p in SCOUT_DIR.iterdir()), "| escaped:", Path("../data/escaped.txt").exists())

mars_day.txt           False | "saved 12 characters to mars_day.txt"
../escaped.txt         True | ValueError: '../escaped.txt' is outside Scout's folder. Use a plain file na
/tmp/escaped.txt       True | ValueError: '/tmp/escaped.txt' is outside Scout's folder. Use a plain file 
a/../../escaped.txt    True | ValueError: 'a/../../escaped.txt' is outside Scout's folder. Use a plain fi
mars_day.txt           True | FileExistsError: 'mars_day.txt' already exists. Pick a new name or set over
folder: ['mars_day.txt'] | escaped: False


### 7. Scout's new tools in the loop
The same fake-model run as lesson 27, on the redesigned tools: search, calculate, save, answer.

In [20]:
QUESTION = "How many minutes long is a day on Mars? Save the answer to mars_minutes.txt."
def last_result(messages): return json.loads(messages[-1]["content"])

SCRIPT = [
    {"tool_calls": [{"name": "search_notes", "args": {"query": "day on Mars"}}]},
    {"tool_calls": [{"name": "calculator", "args": {"expression": "24*60+39"}}]},
    lambda m: {"tool_calls": [{"name": "write_file", "args": {
        "filename": "mars_minutes.txt", "text": f"A day on Mars lasts about {last_result(m)} minutes."}}]},
    {"text": "A day on Mars lasts about 1479 minutes. I saved that to mars_minutes.txt."},
]
run = run_agent(QUESTION, ScriptedModel(SCRIPT), registry.functions, registry.schemas)
print(run["stop"], "| errors:", run["errors"], "|", (SCOUT_DIR / "mars_minutes.txt").read_text())

step 1: search_notes({'query': 'day on Mars'}) -> {"total": 1, "page": 1, "results": [{"id": 3, "text": "A day on Mars l
step 2: calculator({'expression': '24*60+39'}) -> 1479
step 3: write_file({'filename': 'mars_minutes.txt', 'text': 'A day on Mars lasts about 1479 minutes.'}) -> "saved 39 characters to mars_minutes.txt"
step 4: answer
end_of_turn | errors: 0 | A day on Mars lasts about 1479 minutes.


### 8. What tool definitions cost
Every schema is sent as input on **every** model call. Let's compare three tool sets: `do_stuff`
alone, Scout's three good tools, and a "kitchen sink" of 20 tools (Scout's 3 plus 17 near-duplicates,
which is how tool lists grow in real projects). First a rough offline guess: about 4 characters per
token for English (lesson 03).

In [21]:
filler = ("Look up notes by one field and return the matching notes with every field included, "
          "sorted by date, for use when the user asks about that field.")
SINK = registry.schemas + [
    {"name": f"notes_by_field_{i}", "description": filler,
     "parameters": {"type": "object", "required": ["value"],
                    "properties": {"value": {"type": "string", "description": "the value to match"}}}}
    for i in range(17)]
TOOL_SETS = {"do_stuff": [schema_from_function(do_stuff)], "scout (3)": registry.schemas, "sink (20)": SINK}
for name, schemas in TOOL_SETS.items():
    print(f"{name:10} {len(json.dumps(schemas)):6} characters ~ {len(json.dumps(schemas)) // 4:5} tokens")

do_stuff      203 characters ~    50 tokens
scout (3)    1502 characters ~   375 tokens
sink (20)    7153 characters ~  1788 tokens


Now the real count. Gemini's `count_tokens` (lesson 12) accepts `tools` in its config and is free.
We count the question alone, then with each tool set. The difference is what the tools add.

In [22]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)

def declarations(schemas: list[dict]) -> list[types.Tool]:
    return [types.Tool(function_declarations=[types.FunctionDeclaration(
        name=s["name"], description=s["description"], parameters_json_schema=s["parameters"])
        for s in schemas])]

The counts. If this cell is skipped (no recording, no key), the offline estimate above is your guide.

In [23]:
base = client.models.count_tokens(model=MODEL, contents=QUESTION).total_tokens
print(f"question alone: {base} tokens")
for name, schemas in TOOL_SETS.items():
    total = client.models.count_tokens(model=MODEL, contents=QUESTION,
                                       config=types.CountTokensConfig(tools=declarations(schemas))).total_tokens
    print(f"{name:10} adds {total - base:5} tokens per call | x 3 calls per run = {3 * (total - base):5}")

question alone: 20 tokens
do_stuff   adds     8 tokens per call | x 3 calls per run =    24
scout (3)  adds   131 tokens per call | x 3 calls per run =   393
sink (20)  adds   767 tokens per call | x 3 calls per run =  2301


Read the numbers above. The 4-characters rule overestimated JSON by about 3x (375 guessed, 131 real
for Scout's tools): estimate offline, measure with count_tokens, and bill from the call's usage.
`do_stuff` is the cheapest, and useless: a short schema is not a good one.
The 20-tool list costs several times what Scout's three tools cost, on every call, before the model
reads a word of the question. And more tools is not more ability: 17 near-identical tools give the
model more ways to pick the wrong one ("more tools don't always lead to better outcomes", Anthropic).

### 9. Gemini uses the new tools
Lesson 28's `to_contents` and `GeminiModel`. One small change: a truncated result is plain text,
not JSON, so `as_output` passes it through as a string. The model also records its input tokens.

In [24]:
def as_output(content: str):
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return content                                          # truncated text stays text

def to_contents(messages: list[dict]) -> list[types.Content]:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])
        else:
            response = {"error": m["content"]} if m["is_error"] else {"output": as_output(m["content"])}
            part = types.Part(function_response=types.FunctionResponse(
                id=m["call_id"], name=m["name"], response=response))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

`GeminiModel` from lesson 28, plus a list of `prompt_token_count` values, one per call.

In [25]:
SYSTEM = ("You are Scout. Answer questions about the user's notes. Search the notes first. "
          "Use the calculator for arithmetic. If a tool returns an error, fix the call and try again. "
          "Answer in one sentence.")

class GeminiModel:
    def __init__(self):
        self.prompt_tokens = []

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        cfg = types.GenerateContentConfig(system_instruction=SYSTEM, max_output_tokens=400,
                                          tools=declarations(tools), automatic_function_calling=NO_AFC)
        resp = client.models.generate_content(model=MODEL, contents=to_contents(messages), config=cfg)
        self.prompt_tokens.append(resp.usage_metadata.prompt_token_count)
        raw = resp.candidates[0].content
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        if calls:
            return {"role": "assistant", "tool_calls": calls, "raw": raw}
        return {"role": "assistant", "text": resp.text, "raw": raw}

Scout answers the same question with Gemini choosing every call. We first count the first call's input
with `count_tokens` (system prompt + tools + question), then compare it with what the real call billed.

In [26]:
ASK = "How many minutes long is a day on Mars? Save the answer to mars_gemini.txt."
counted = client.models.count_tokens(model=MODEL, contents=ASK, config=types.CountTokensConfig(
    system_instruction=SYSTEM, tools=declarations(registry.schemas))).total_tokens
gemini = GeminiModel()
run = run_agent(ASK, gemini, registry.functions, registry.schemas)
print(run["stop"], "| errors:", run["errors"], "|", run["answer"])
print("input tokens per call:", gemini.prompt_tokens, "| count_tokens for call 1:", counted)
print("saved:", sorted(p.name for p in SCOUT_DIR.iterdir()))

step 1: search_notes({'query': 'Mars'}) -> {"total": 34, "page": 1, "results": [{"id": 1, "text": "Mars has two s
step 2: calculator({'expression': '24 * 60 + 39'}) -> 1479
step 3: write_file({'filename': 'mars_gemini.txt', 'text': 'A day on Mars is 1479 minutes long.', 'overwrite': True}) -> "saved 35 characters to mars_gemini.txt"
step 4: answer
end_of_turn | errors: 0 | A day on Mars is 1479 minutes long, and I have saved this answer to mars_gemini.txt.
input tokens per call: [341, 477, 503, 553] | count_tokens for call 1: 195
saved: ['mars_day.txt', 'mars_gemini.txt', 'mars_minutes.txt']


Read the trace: Gemini searched, got page 1 of 34 matches (the day note is on it), used the
calculator, and saved with a plain name. It set overwrite on its own (see `'overwrite': True` in step 3):
models do set optional flags. Input grew from 341 to 553 tokens as results piled up, so small results
matter. `count_tokens` said 195 for call 1, but the call billed 341. The two numbers differ (the provider
adds text we cannot see), so check cost against the `usage` of real calls.

## At work
- **Fewer, consolidated tools.** Anthropic's "Writing tools for agents" advises a `search_logs` that
  returns relevant lines over a `read_logs` that returns everything; one small description fix
  stopped Claude adding "2025" to web searches. Teams prove such changes with an eval. → lesson 48
- **Hard caps on result size.** Claude Code limits tool responses to 25,000 tokens by default; Anthropic
  measured a "concise" result at 72 tokens against 206 for "detailed". → lesson 31
- **Tool definitions are billed on every call.** Anthropic adds a 286-token tool-use system prompt on
  Sonnet 5.5 and Opus 5.5 on top of your schemas, so teams track it per run and prune unused
  tools. → lesson 54
- **Least privilege by construction**: AST calculators instead of `eval`, file tools confined to one
  folder, and human sign-off for side effects; tool text itself is treated as untrusted. → lessons 51, 52

## What just happened
- `do_stuff` hid three jobs behind two strings, returned 4,300 characters for one fact, ran `eval` on
  model text and could write anywhere.
- `search_notes` does one job, says what it returns, clamps `max_results`, shortens long notes and
  pages results with `total` and `next_page`.
- A cap in `run_tool` truncates any oversized result and tells the model how to ask for less.
- The AST `calculator` evaluated arithmetic and refused code and an exponent bomb as error results.
- `write_file` resolves the path and refuses anything outside Scout's folder, and won't overwrite by default.
- Tool definitions were counted: more tools means more input tokens on every call.

## Common mistakes
**1. Checking the folder with `startswith`.** It looks safe, but a sibling folder whose name *starts*
with `scout_files` passes the check. The assert shows the escape.

In [27]:
def is_inside_naive(filename: str) -> bool:
    return str((SCOUT_DIR / filename).resolve()).startswith(str(SCOUT_DIR.resolve()))

name = "../scout_files_backup/notes.txt"
assert not is_inside_naive(name), f"{name!r} passed the check but lands in {Path(name).parts[1]}/"

AssertionError: '../scout_files_backup/notes.txt' passed the check but lands in scout_files_backup/

Fix: compare whole paths, not strings. `Path.is_relative_to` (or `write_file`'s parent check) treats
`scout_files_backup` as a different folder.

In [28]:
target = (SCOUT_DIR / name).resolve()
print("inside Scout's folder:", target.is_relative_to(SCOUT_DIR.resolve()))

inside Scout's folder: False


**2. Using `ast.literal_eval` as the calculator.** It is safe, but it only reads literal values, not
sums like `24*60`, so the first real calculation fails.

In [29]:
import re
try:
    ast.literal_eval("24*60+39")
except ValueError as e:                     # strip the memory address so the message is stable
    raise ValueError(re.sub(r" object at 0x[0-9a-f]+", "", str(e))) from None

ValueError: malformed node or string on line 1: <ast.BinOp>

Fix: walk the tree yourself with an allow-list of operators, as `calculator` does.

In [30]:
print(calculator("24*60+39"))

1479


## Try it
Anthropic suggests letting the model choose how much detail it gets back. Write `search_notes_v2`
with a `detail: str = "short"` parameter: `"short"` returns only `id` and the first 40 characters of
each note, `"full"` returns the full text. Raise an actionable `ValueError` for any other value. Print
`len(json.dumps(...))` for `"Mars"` in both modes, and check the schema has `detail` but not in `required`.

In [31]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [32]:
llm_replay.stop()

## Key terms
- **tool design** — shaping a tool for the model as its user: one job, clear name and description, narrow inputs, small results.
- **high-signal result** — a tool result holding only what helps the next step, not a raw dump.
- **pagination** — returning results one page at a time with a total and a next-page marker, so the model asks for more only when needed.
- **truncation (tool results)** — cutting a result past a size limit, with a note saying so.
- **poka-yoke** — designing a tool so misuse is impossible or harmless, e.g. clamping a page size.
- **safe evaluation** — computing a model's expression by walking its syntax tree with an allow-list, never `eval`.
- **path traversal** — escaping a folder with names like `../x` or absolute paths; stopped by resolving the path first.
- **tool-definition token cost** — the input tokens every tool schema adds to every model call, whether or not the tool is used.